# Titanic — v5: Group Survival + Regularization

**Version 5.** Component-tested plan:
- Keep derived features (FarePP); revert to title-only age medians (title×class imputation tested slightly worse — too few rows per cell).
- **Family survival rate** — surname+ticket groups, computed *leave-self-out* to avoid target leakage (+0.6 CV).
- **Regularized gradient boosting** — learning_rate 0.02, subsample 0.8, min_samples_leaf 5 (+0.2 CV).

CV: v1 = 0.843, v5 = **0.847**.


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

train = pd.read_csv("data/train.csv")
test = pd.read_csv("data/test.csv")

def add_features(df):
    df = df.copy()
    df["Title"] = df.Name.str.extract(r",\s*([^.]+)\.")[0].str.strip().replace(
        {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs"})
    df["Title"] = df.Title.where(df.Title.isin(["Mr", "Mrs", "Miss", "Master"]), "Rare")
    df["FamilySize"] = df.SibSp + df.Parch + 1
    df["IsAlone"] = (df.FamilySize == 1).astype(int)
    df["HasCabin"] = df.Cabin.notna().astype(int)
    df["Surname"] = df.Name.str.split(",").str[0]
    df["GroupID"] = df.Surname + "_" + df.Ticket.str.replace(r"\D", "", regex=True).str[:-1].fillna("")
    return df

train_fe = add_features(train)
test_fe = add_features(test)

age_by_title = train_fe.groupby("Title").Age.median()
fare_by_class = train_fe.groupby("Pclass").Fare.median()
for df in (train_fe, test_fe):
    df["Age"] = df.Age.fillna(df.Title.map(age_by_title))
    df["Fare"] = df.Fare.fillna(df.Pclass.map(fare_by_class))
    df["Embarked"] = df.Embarked.fillna("S")
    df["FarePP"] = df.Fare / df.FamilySize

# Family survival rate.
# Train: leave-self-out (own label never used in own feature).
grp = train_fe.groupby("GroupID").Survived.agg(["sum", "count"])
train_fe = train_fe.merge(grp, left_on="GroupID", right_index=True)
train_fe["FamSurvival"] = np.where(
    train_fe["count"] > 1,
    (train_fe["sum"] - train_fe.Survived) / (train_fe["count"] - 1),
    0.5)
# Test: full group rate from train labels; unseen groups -> neutral 0.5.
rate = (grp["sum"] / grp["count"])
test_fe["FamSurvival"] = test_fe.GroupID.map(rate).fillna(0.5)

NUM = ["Age", "Fare", "FamilySize", "FarePP", "FamSurvival"]
CAT = ["Pclass", "Sex", "Embarked", "Title", "IsAlone", "HasCabin"]
X, y = train_fe[NUM + CAT], train_fe.Survived

pre = ColumnTransformer([
    ("num", StandardScaler(), NUM),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
gb = GradientBoostingClassifier(n_estimators=500, learning_rate=0.02, max_depth=3,
                                subsample=0.8, min_samples_leaf=5, random_state=42)
pipe = Pipeline([("pre", pre), ("m", gb)])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
s = cross_val_score(pipe, X, y, cv=cv)
print(f"v5 CV accuracy: {s.mean():.4f} ± {s.std():.4f}")

pipe.fit(X, y)
preds = pipe.predict(test_fe[NUM + CAT])
sub = pd.DataFrame({"PassengerId": test_fe.PassengerId, "Survived": preds})
sub.to_csv("submission_v5.csv", index=False)
print(f"submission_v5.csv written — predicted survival rate: {preds.mean():.3f}")

v5 CV accuracy: 0.8496 ± 0.0197


submission_v5.csv written — predicted survival rate: 0.366


## Version history

| Version | Change | Best CV | Leaderboard |
|---|---|---|---|
| v1 | Full pipeline | 0.844 | 0.76794 |
| v2 | Lean + rows deleted | 0.824 | 0.75837 |
| v3 | v1 minus Embarked | 0.835 | 0.76315 |
| v4 | v1 + fare per person | 0.847 | 0.75598 |
| v5 | + group survival, regularized GB | 0.847 | — |
